In [1]:
import os, sys, importlib
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
from pathlib import Path
import yaml

pd.set_option('display.max_columns', None)
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
config_path = Path(module_path) / "source_file.yml"
with open(config_path, "r", encoding="utf-8") as f:
    config_data = yaml.safe_load(f)

layer_key = config_data.get("gis_layer", "baseline")
shp_path = config_data.get("simulation_shp_files", {}).get(layer_key)
if not shp_path:
    raise ValueError(f"No GIS layer path configured for '{layer_key}' in {config_path}")

dataset_name: str = config_data.get("bronze_dataset_name", "bronze")
table_name: str = f"gis_layer_{layer_key}"
md_db: str = config_data.get("motherduck_db", "sumo_visualization")

motherduck_token = os.getenv("MOTHERDUCK_TOKEN")
md_path = f"md:{md_db}"
if motherduck_token:
    md_path = f"{md_path}?motherduck_token={motherduck_token}"

con = duckdb.connect(md_path)
print(f"Connected to MotherDuck DB: {md_db}")
print(f"Layer source: {shp_path}")
print(f"Target table: {dataset_name}.{table_name}")

Connected to MotherDuck DB: sumo_visualization
Layer source: /Users/mmh/Library/CloudStorage/OneDrive-OakRidgeNationalLaboratory/nashville_microsim/Model/SUMO/shp/links_sumo_wcc.parquet
Target table: bronze.gis_layer_baseline


In [3]:
path_obj = Path(shp_path)
if not path_obj.exists():
    raise FileNotFoundError(f"GIS layer not found: {path_obj}")

if path_obj.suffix.lower() in {".parquet", ".geoparquet"}:
    try:
        gdf = gpd.read_parquet(path_obj)
    except Exception:
        # Fallback for non-GeoParquet files: try to reconstruct geometry from common columns.
        pdf = pd.read_parquet(path_obj)
        geom_col = next((c for c in ["geometry", "geom", "geometry_wkt", "wkt"] if c in pdf.columns), None)

        if geom_col is not None:
            series = pdf[geom_col]
            non_null = series.dropna()
            sample = non_null.iloc[0] if len(non_null) else None

            if isinstance(sample, (bytes, bytearray, memoryview)):
                geom = gpd.GeoSeries.from_wkb(series)
            else:
                geom = gpd.GeoSeries.from_wkt(series.astype(str))
            
            pdf = pdf.drop(columns=[geom_col])
            gdf = gpd.GeoDataFrame(pdf, geometry=geom)
            
            # Heuristic CRS detection
            first_geom = geom.iloc[0] if not geom.empty else None
            if first_geom and abs(first_geom.bounds[0]) > 180:
                gdf.crs = "EPSG:3857"
            else:
                gdf.crs = "EPSG:4326"
        else:
            gdf = gpd.GeoDataFrame(pdf)
else:
    gdf = gpd.read_file(path_obj)

if gdf.empty:
    raise ValueError("GIS layer loaded successfully, but it contains no rows")

# Standardize to EPSG:4326 for Folium
has_geometry = "geometry" in gdf.columns and gdf.geometry.notna().any()
if has_geometry:
    if gdf.crs is None:
        # Fallback if no CRS info
        first_geom = gdf.geometry.iloc[0]
        if first_geom and abs(first_geom.bounds[0]) > 180:
            gdf.crs = "EPSG:3857"
        else:
            gdf.crs = "EPSG:4326"
    
    if str(gdf.crs).upper() != "EPSG:4326":
        print(f"Reprojecting from {gdf.crs} to EPSG:4326")
        gdf = gdf.to_crs(epsg=4326)

upload_df = gdf.copy()
if has_geometry:
    upload_df["geometry_wkt"] = upload_df.geometry.to_wkt()
    upload_df = pd.DataFrame(upload_df.drop(columns="geometry"))
else:
    upload_df = pd.DataFrame(upload_df)

safe_dataset = '"' + dataset_name.replace('"', '""') + '"'
safe_table = '"' + table_name.replace('"', '""') + '"'
full_table = f"{safe_dataset}.{safe_table}"

con.register("gis_upload_df", upload_df)
con.execute(f"CREATE SCHEMA IF NOT EXISTS {safe_dataset}")
con.execute(f"CREATE OR REPLACE TABLE {full_table} AS SELECT * FROM gis_upload_df")
con.unregister("gis_upload_df")

print(f"Uploaded {len(upload_df):,} rows to {dataset_name}.{table_name}")
if has_geometry:
    print("Geometry exported as WKT in column: geometry_wkt")
print(f"Columns: {', '.join(upload_df.columns)}")

Reprojecting from EPSG:3857 to EPSG:4326


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Uploaded 164,783 rows to bronze.gis_layer_baseline
Geometry exported as WKT in column: geometry_wkt
Columns: ID, REF_IN_ID, NREF_IN_ID, speed_mps, priority, FUNC_CLASS, DIR, numLanes, RTE_NME, LENGTH, geometry_wkt
